In [1]:
import torch
torch.cuda.empty_cache()
torch._C._cuda_getDeviceCount()

1

In [2]:
print(torch.version.cuda)   
print(torch.cuda.is_available())

12.1
True


In [ ]:
import requests
# Call pdf retrieval API, then proceed to chunking

print(f"Number of chunks: {len(doc_chunks)}")
print("\n == CHUNKS ==\n")
print(*doc_chunks[150:153], sep="\n\n")

Number of chunks: 568

 == CHUNKS ==

rate and reduce symptoms.467–469
IIa
B
Atrioventricular node ablation combined with 
cardiac resynchronization therapy should be 
considered in severely symptomatic patients with 
permanent AF and at least one hospitalization for HF 
to reduce symptoms, physical limitations, recurrent 
HF hospitalization, and mortality.470,471
IIa
B
Intravenous amiodarone, digoxin, esmolol, or 
landiolol may be considered in patients with AF who 
have haemodynamic instability or severely depressed 
LVEF to achieve acute control of heart rate.472,473
IIb
B
© ESC 2024
AF, atrial fibrillation; b.p.m., beats per minute; HF, heart failure; LVEF, left ventricular 
ejection fraction. 
aClass of recommendation. 
bLevel of evidence.
Recommendation Table 13 — Recommendations for 
management of bleeding in anticoagulated patients 
(see also Evidence Table 13)
Recommendations
Classa
Levelb
Interrupting anticoagulation and performing 
diagnostic or treatment interventions is 
r

In [ ]:
# Call embedding

vector_db = [(chunk, vector) for chunk,vector in zip(doc_chunks, emb_docs)]
vector_db[0]

('2024 ESC Guidelines for the management \nof atrial fibrillation developed in collaboration \nwith the European Association \nfor Cardio-Thoracic Surgery (EACTS)\nDeveloped by the task force for the management of atrial fibrillation of the \nEuropean Society of Cardiology (ESC), with the special contribution of the \nEuropean Heart Rhythm Association (EHRA) of the ESC.  \nEndorsed by the European Stroke Organisation (ESO)\nAuthors/Task Force Members: Isabelle C. Van Gelder  *†, (Chairperson) \n(Netherlands), Michiel Rienstra  \n±, (Task Force Co-ordinator) (Netherlands), \nKarina V. Bunting  \n±, (Task Force Co-ordinator) (United Kingdom), \nRuben Casado-Arroyo  \n(Belgium), Valeria Caso  \n1 (Italy), Harry J.G.M. Crijns  \n(Netherlands), Tom J.R. De Potter  \n(Belgium), Jeremy Dwight (United Kingdom), \nLuigina Guasti  \n(Italy), Thorsten Hanke  \n2 (Germany), Tiny Jaarsma  \n(Sweden), \nMaddalena Lettino  \n(Italy), Maja-Lisa Løchen  \n(Norway), R. Thomas Lumbers  \n(United Kingdom)

In [8]:
ex_query = "please write a discharge letter to a patient presenting atrial fibrillation, male over 65 with migraines?"

retrieved_knowledge = retrieval(ex_query, vector_db, emb_model, top_n=5)
print(*retrieved_knowledge, sep="\n")

('Keywords \nGuidelines • Atrial fibrillation • AF-CARE • Comorbidity • Risk factors • Anticoagulation • Rate control • Rhythm \ncontrol • Cardioversion • Antiarrhythmic drugs • Catheter ablation • AF surgery • Evaluation • Stroke • \nThromboembolism\nESC Guidelines                                                                                                                                                                                          3315\nDownloaded from https://academic.oup.com/eurheartj/article/45/36/3314/7738779 by guest on 29 May 2025', 0.717186857978088)
('77. Zuin M, Roncon L, Passaro A, Bosi C, Cervellati C, Zuliani G. Risk of dementia in pa\xad\ntients with atrial fibrillation: short versus long follow-up. A systematic review and \nmeta-analysis. Int J Geriatr Psychiatry 2021;36:1488–500. https://doi.org/10.1002/gps. \n5582\n78. Mobley AR, Subramanian A, Champsi A, Wang X, Myles P, McGreavy P, et al. \nThromboembolic events and vascular dementia in patients with 

In [ ]:
# reranking

{'corpus_id': 4, 'score': np.float32(-5.889958), 'text': '264. Kim D, Yang PS, Kim TH, Jang E, Shin H, Kim HY, et al. Ideal blood pressure in patients \nwith atrial fibrillation. J Am Coll Cardiol 2018;72:1233–45. https://doi.org/10.1016/j.jacc. \n2018.05.076\n265. Lip GY, Clementy N, Pericart L, Banerjee A, Fauchier L. Stroke and major bleeding risk \nin elderly patients aged ≥75 years with atrial fibrillation: the Loire valley atrial fibrilla\xad\ntion project. Stroke 2015;46:143–50. https://doi.org/10.1161/STROKEAHA.114. \n007199\n266. American Diabetes Association Professional Practice Committee. 2. Classification and \ndiagnosis of diabetes: standards of medical care in diabetes—2022. Diabetes Care 2022; \n45:S17–38. https://doi.org/10.2337/dc22-S002\n267. Steensig K, Olesen KKW, Thim T, Nielsen JC, Jensen SE, Jensen LO, et al. Should the \npresence or extent of coronary artery disease be quantified in the CHA2DS2-VASc \nscore in atrial fibrillation? A report from the western Denm

In [17]:
# login to te hf hub to access the model
from huggingface_hub import notebook_login

notebook_login()

In [25]:
# combining retrived knowledge into the prompt
context = '\n'.join([f" - {chunk['text']}" for chunk in reranked_knowledge])
sys_prompt = f"""You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information. 
You aid medical professionals in writing dismissal letters from the ER. 
Discharge letters should be concise, clear, and contain all necessary information for the patient's follow-up care, comprising treatment and prescription, which you can take from the context.
The discharge letter should be written in a professional tone, suitable for a medical setting.
The discharge letter should contain a section for the patient treatment, where specific medications and dosages are mentioned.
Medications and dosages should be taken from the context.
Context:
{context}"""

messages = [
    {"role": "system", "content": sys_prompt},
    {"role": "user", "content": ex_query},
]

formatted_chat = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
formatted_chat_text = tokenizer.decode(formatted_chat[0])
# print(f"\nFORMATTED PROMTP:\n\n{formatted_chat_text}")

# generating response
output = generate_response(formatted_chat, model, tokenizer)
output_text = output[len(formatted_chat_text):].strip("<|eot_id|>")
print(f"\nRESPONSE:\n\n{output_text}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



RESPONSE:

[Your Name]
[Your Title]
[Medical Facility]
[Date]

[Patient's Name]
[Patient's Address]

Dear [Patient's Name],

Re: Discharge Letter for Atrial Fibrillation

It is our pleasure to inform you that you have been discharged from our hospital, effective [Date of Discharge]. We are pleased to have had the opportunity to care for you during your stay.

As you prepare to return home, we would like to take a moment to review the key points of your care and provide you with the necessary information to ensure your safe transition back into your daily routine.

**Treatment and Medications**

We have reviewed your treatment plan and have confirmed that you will be taking the following medications:

* [Name of
